<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/XGBoost2WWMultiDatasetHyperparameterSweep.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# xgboost2ww Multi-Dataset Hyperparameter Sweep (10 Tabular Benchmarks)

This notebook extends `XGBoost2WWAdultIncomeHyperparameterSweep` to run **10 tabular datasets** with mixed categorical/numeric preprocessing, checkpointed reruns, and WeightWatcher conversion.

For each dataset, it:
1. Loads and preprocesses data.
2. Trains a controlled XGBoost hyperparameter sweep.
3. Saves checkpointed per-run results.
4. Produces **one set of 3 plots**:
   - alpha vs test accuracy
   - alpha vs generalization gap
   - ERG gap vs test accuracy

## 1) Mount Google Drive and configure checkpoint paths

In [ ]:
from pathlib import Path
from datetime import datetime, UTC
import json
import os
import logging
import time

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    pass

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/xgboost2ww_checkpoints/multidataset_sweep')
else:
    CHECKPOINT_DIR = Path('./checkpoints/multidataset_sweep')

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_ID = datetime.now(UTC).strftime('xgbww_multidataset_%Y%m%dT%H%M%SZ')
RESULTS_CSV = CHECKPOINT_DIR / 'multidataset_hyperparameter_results.csv'
CONFIG_JSON = CHECKPOINT_DIR / f'{EXPERIMENT_ID}_config.json'

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('xgbww_multidataset')
logger.info('Checkpoint directory: %s', CHECKPOINT_DIR.resolve())


## 2) Install required packages (Colab)

In [ ]:
!pip -q install xgboost weightwatcher xgboost2ww

## 3) Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import xgboost as xgb
import weightwatcher as ww
from xgboost2ww import convert

## 4) Reproducibility + run settings

In [ ]:
RNG = 123
np.random.seed(RNG)

TEST_SIZE = 0.2
NFOLDS = 5
T_POINTS = 80
MAX_DENSE_ELEMENTS = int(2e8)
MIN_CLASS_COUNT = 2

RESTART_EXPERIMENT = False  # set True to start fresh and overwrite previous checkpoint CSV

## 5) Dataset catalog (10 datasets)

In [ ]:
# Notes:
# - These datasets are selected for strong signal + sufficient complexity for WW alpha/ERG analysis.
# - OpenML IDs are preferred for reproducibility when known.
DATASETS = [
    dict(key='adult', label='Adult Income', openml_id=1590, openml_name='adult', target='class', task='binary'),
    dict(key='bank_marketing', label='Bank Marketing', openml_id=1461, openml_name='bank-marketing', target='y', task='binary'),
    dict(key='credit_g', label='German Credit', openml_id=31, openml_name='credit-g', target='class', task='binary'),
    dict(key='default_credit_card', label='Default Credit Card', openml_id=42477, openml_name='default-of-credit-card-clients', target='default payment next month', task='binary'),
    dict(key='amazon_employee_access', label='Amazon Employee Access', openml_id=4135, openml_name='Amazon_employee_access', target='target', task='binary'),
    dict(key='qsar_biodeg', label='QSAR Biodeg', openml_id=1494, openml_name='qsar-biodeg', target='Class', task='binary'),
    dict(key='phoneme', label='Phoneme', openml_id=1489, openml_name='phoneme', target='Class', task='binary'),
    dict(key='spambase', label='Spambase', openml_id=44, openml_name='spambase', target='class', task='binary'),
    dict(key='eeg_eye_state', label='EEG Eye State', openml_id=1471, openml_name='eeg-eye-state', target='Class', task='binary'),
    dict(key='covertype', label='Covertype', openml_id=1596, openml_name='covertype', target='class', task='multiclass'),
]

DATASET_KEYS_TO_RUN = [d['key'] for d in DATASETS]  # customize subset if needed


## 6) Hyperparameter sweep configs (shared template)

In [ ]:
COMMON_BASE = dict(
    tree_method='hist',
    seed=RNG,
)

CONFIGS = [
    dict(name='A_strong_reg', max_depth=3, learning_rate=0.04, subsample=0.80, colsample_bytree=0.75,
         reg_lambda=8.0, reg_alpha=1.0, min_child_weight=8.0, gamma=1.5, num_boost_round=1000, early_stopping_rounds=80),
    dict(name='B_balanced', max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.80,
         reg_lambda=5.0, reg_alpha=0.5, min_child_weight=5.0, gamma=1.0, num_boost_round=900, early_stopping_rounds=70),
    dict(name='C_standard', max_depth=5, learning_rate=0.06, subsample=0.90, colsample_bytree=0.90,
         reg_lambda=3.0, reg_alpha=0.2, min_child_weight=3.0, gamma=0.5, num_boost_round=800, early_stopping_rounds=60),
    dict(name='D_aggressive', max_depth=6, learning_rate=0.08, subsample=0.95, colsample_bytree=1.00,
         reg_lambda=1.0, reg_alpha=0.0, min_child_weight=1.0, gamma=0.0, num_boost_round=700, early_stopping_rounds=50),
]

## 7) Helper functions

In [ ]:
def robust_fetch_openml(spec, max_retries=3, retry_wait_s=5):
    errs = []

    for source in [('id', spec.get('openml_id')), ('name', spec.get('openml_name'))]:
        kind, value = source
        if value is None:
            continue
        for attempt in range(1, max_retries + 1):
            try:
                logger.info('Loading dataset %s via OpenML %s=%s (attempt %d/%d)', spec['key'], kind, value, attempt, max_retries)
                t0 = time.time()
                if kind == 'id':
                    result = fetch_openml(data_id=value, as_frame=True, parser='auto')
                else:
                    result = fetch_openml(name=value, as_frame=True, parser='auto')
                logger.info('Loaded dataset %s via %s=%s in %.1fs', spec['key'], kind, value, time.time() - t0)
                return result
            except Exception as e:
                msg = f"{kind}={value} attempt={attempt}: {e}"
                errs.append(msg)
                logger.warning('OpenML load failed for %s: %s', spec['key'], msg)
                if attempt < max_retries:
                    time.sleep(retry_wait_s)

    raise RuntimeError(' | '.join(errs))


def to_binary_target(y_raw):
    s = pd.Series(y_raw)
    if s.dtype.kind in 'biu' and s.nunique(dropna=True) == 2:
        vals = sorted(s.dropna().unique().tolist())
        return s.map({vals[0]: 0, vals[1]: 1}).astype(int).to_numpy(), {vals[0]: 0, vals[1]: 1}

    normalized = s.astype(str).str.strip().str.lower()
    pos_tokens = {'1', 'true', 'yes', 'y', 'churn', '>50k', 'default', 'bad', 'attrition', 'positive'}
    if normalized.nunique(dropna=True) == 2:
        vals = sorted(normalized.dropna().unique().tolist())
        if vals[1] in pos_tokens:
            mapping = {vals[0]: 0, vals[1]: 1}
        elif vals[0] in pos_tokens:
            mapping = {vals[0]: 1, vals[1]: 0}
        else:
            mapping = {vals[0]: 0, vals[1]: 1}
        return normalized.map(mapping).astype(int).to_numpy(), mapping

    cats = pd.Categorical(s)
    if len(cats.categories) != 2:
        raise ValueError(f'Expected binary target, got {len(cats.categories)} classes')
    mapping = {cats.categories[0]: 0, cats.categories[1]: 1}
    return pd.Series(s).map(mapping).astype(int).to_numpy(), mapping


def preprocess_frame(X_df):
    categorical_cols = X_df.select_dtypes(include=['object', 'category', 'bool']).columns
    numeric_cols = X_df.select_dtypes(exclude=['object', 'category', 'bool']).columns

    pre = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
            ('num', StandardScaler(with_mean=False), numeric_cols),
        ],
        remainder='drop',
    )
    X = pre.fit_transform(X_df)
    if not sparse.issparse(X):
        X = sparse.csr_matrix(X)
    return X, pre, list(categorical_cols), list(numeric_cols)


def build_xgb_params(task, n_classes, cfg):
    params = dict(COMMON_BASE)
    if task == 'multiclass':
        params.update(dict(objective='multi:softprob', eval_metric='mlogloss', num_class=n_classes))
    else:
        params.update(dict(objective='binary:logistic', eval_metric='logloss'))

    for k in ['max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'reg_lambda', 'reg_alpha', 'min_child_weight', 'gamma']:
        params[k] = cfg[k]
    return params


def safe_convert_to_w7(model, X_train, y_train, params, best_round, task):
    X_for_convert = X_train
    try:
        return convert(
            model=model,
            data=X_for_convert,
            labels=y_train,
            W='W7',
            nfolds=NFOLDS,
            t_points=T_POINTS,
            random_state=RNG,
            train_params=params,
            num_boost_round=best_round,
            multiclass=('avg' if task == 'multiclass' else 'error'),
        )
    except Exception as sparse_err:
        logger.warning('Sparse convert failed; considering dense fallback. error=%s', sparse_err)
        if sparse.issparse(X_train) and X_train.size <= MAX_DENSE_ELEMENTS:
            X_for_convert = X_train.toarray()
            logger.info('Retrying convert with dense array of shape=%s', X_for_convert.shape)
            return convert(
                model=model,
                data=X_for_convert,
                labels=y_train,
                W='W7',
                nfolds=NFOLDS,
                t_points=T_POINTS,
                random_state=RNG,
                train_params=params,
                num_boost_round=best_round,
                multiclass=('avg' if task == 'multiclass' else 'error'),
            )
        raise

def analyze_w7_with_weightwatcher(layer):
    watcher = ww.WeightWatcher(model=layer)
    details_df = watcher.analyze(ERG=True, randomize=True, plot=False)

    alpha = float(details_df['alpha'].iloc[0]) if 'alpha' in details_df.columns else np.nan
    erg_gap = float(details_df['ERG_gap'].iloc[0]) if 'ERG_gap' in details_df.columns else np.nan
    return alpha, erg_gap


## 8) Save experiment config

In [ ]:
experiment_config = {
    'created_utc': datetime.now(UTC).isoformat(),
    'experiment_id': EXPERIMENT_ID,
    'datasets': DATASETS,
    'dataset_keys_to_run': DATASET_KEYS_TO_RUN,
    'test_size': TEST_SIZE,
    'nfolds': NFOLDS,
    't_points': T_POINTS,
    'max_dense_elements': MAX_DENSE_ELEMENTS,
    'restart_experiment': RESTART_EXPERIMENT,
    'num_configs': len(CONFIGS),
    'configs': CONFIGS,
    'results_csv': str(RESULTS_CSV),
}
with open(CONFIG_JSON, 'w') as f:
    json.dump(experiment_config, f, indent=2)
logger.info('Config saved: %s', CONFIG_JSON)


## 9) Load existing checkpoint

In [ ]:
if RESTART_EXPERIMENT and RESULTS_CSV.exists():
    RESULTS_CSV.unlink()

if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    print(f'Loaded checkpoint with {len(results_df)} rows')
else:
    results_df = pd.DataFrame()
    print('No checkpoint found. Starting fresh.')

completed_keys = set()
if len(results_df):
    completed_keys = set((results_df['dataset_key'] + '::' + results_df['model']).astype(str).tolist())

## 10) Train all datasets × hyperparameter configs with per-run checkpointing

In [ ]:
all_rows = [] if results_df.empty else results_df.to_dict(orient='records')

for spec in DATASETS:
    if spec['key'] not in DATASET_KEYS_TO_RUN:
        logger.info('Skipping dataset %s (not selected)', spec['key'])
        continue

    logger.info('=' * 88)
    logger.info('Dataset: %s (%s)', spec['label'], spec['key'])
    logger.info('=' * 88)

    dataset_start = time.time()
    try:
        raw = robust_fetch_openml(spec)
        df = raw.frame.copy()
        logger.info('Raw dataframe shape for %s: %s', spec['key'], df.shape)

        if spec['target'] in df.columns:
            y_raw = df[spec['target']]
            X_df = df.drop(columns=[spec['target']])
        elif getattr(raw, 'target', None) is not None:
            y_raw = pd.Series(raw.target)
            X_df = raw.frame.copy()
        else:
            raise ValueError(f"Could not resolve target column '{spec['target']}'")

        if spec['task'] == 'binary':
            y, target_mapping = to_binary_target(y_raw)
            n_classes = 2
        else:
            y, uniques = pd.factorize(pd.Series(y_raw))
            target_mapping = {str(u): int(i) for i, u in enumerate(uniques)}
            n_classes = len(uniques)

        if n_classes < MIN_CLASS_COUNT:
            raise ValueError(f'Need at least {MIN_CLASS_COUNT} classes, got {n_classes}')

        X, pre, cat_cols, num_cols = preprocess_frame(X_df)
        logger.info('%s preprocessed: rows=%d raw_features=%d engineered_features=%d cat_cols=%d num_cols=%d',
                    spec['key'], X_df.shape[0], X_df.shape[1], X.shape[1], len(cat_cols), len(num_cols))

        # Complexity checks to keep WW alpha estimates meaningful.
        if X_df.shape[0] < 1000 or X.shape[1] < 20:
            raise ValueError(
                f"Dataset too simple for stable alpha metrics: rows={X_df.shape[0]}, features_after_pre={X.shape[1]}"
            )

        stratify_y = y if n_classes > 1 else None
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RNG, stratify=stratify_y
        )

        dtrain = xgb.DMatrix(X_train, label=y_train)
        dtest = xgb.DMatrix(X_test, label=y_test)

        for cfg in CONFIGS:
            run_key = f"{spec['key']}::{cfg['name']}"
            if run_key in completed_keys:
                logger.info('Skipping completed run: %s', run_key)
                continue

            logger.info('Training %s with config=%s', spec['key'], cfg['name'])
            params = build_xgb_params(spec['task'], n_classes, cfg)
            evals = [(dtrain, 'train'), (dtest, 'test')]

            model = xgb.train(
                params=params,
                dtrain=dtrain,
                num_boost_round=cfg['num_boost_round'],
                evals=evals,
                early_stopping_rounds=cfg['early_stopping_rounds'],
                verbose_eval=False,
            )

            best_round = int(getattr(model, 'best_iteration', cfg['num_boost_round'] - 1) + 1)
            train_pred = model.predict(dtrain)
            test_pred = model.predict(dtest)

            if spec['task'] == 'multiclass':
                y_train_hat = np.argmax(train_pred, axis=1)
                y_test_hat = np.argmax(test_pred, axis=1)
            else:
                y_train_hat = (train_pred >= 0.5).astype(int)
                y_test_hat = (test_pred >= 0.5).astype(int)

            train_acc = float(accuracy_score(y_train, y_train_hat))
            test_acc = float(accuracy_score(y_test, y_test_hat))
            gap = train_acc - test_acc

            layer_w7 = safe_convert_to_w7(model, X_train, y_train, params, best_round, spec['task'])
            alpha, erg_gap = analyze_w7_with_weightwatcher(layer_w7)

            row = {
                'dataset_key': spec['key'],
                'dataset_label': spec['label'],
                'task': spec['task'],
                'n_rows': int(X_df.shape[0]),
                'n_features_raw': int(X_df.shape[1]),
                'n_features_after_pre': int(X.shape[1]),
                'n_categorical_cols': int(len(cat_cols)),
                'n_numeric_cols': int(len(num_cols)),
                'target_mapping': json.dumps(target_mapping),
                'model': cfg['name'],
                'status': 'ok',
                'best_round': best_round,
                'alpha': alpha,
                'ERG_gap': erg_gap,
                'train_acc': train_acc,
                'test_acc': test_acc,
                'generalization_gap': gap,
                **{k: cfg[k] for k in ['max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'reg_lambda', 'reg_alpha', 'min_child_weight', 'gamma']},
            }

            all_rows.append(row)
            completed_keys.add(run_key)
            pd.DataFrame(all_rows).to_csv(RESULTS_CSV, index=False)
            logger.info('Completed %s | acc=%.4f alpha=%.4f ERG_gap=%.4f best_round=%d', run_key, test_acc, alpha, erg_gap, best_round)

        logger.info('Finished dataset %s in %.1fs', spec['key'], time.time() - dataset_start)

    except Exception as e:
        logger.exception('Dataset failed: %s', spec['key'])
        fail_row = {
            'dataset_key': spec['key'],
            'dataset_label': spec['label'],
            'task': spec['task'],
            'model': 'dataset_load_or_train_failure',
            'status': 'failed',
            'error': str(e),
        }
        all_rows.append(fail_row)
        pd.DataFrame(all_rows).to_csv(RESULTS_CSV, index=False)


## 11) Load final results

In [ ]:
results_df = pd.read_csv(RESULTS_CSV)
results_df = results_df.sort_values(['dataset_key', 'test_acc'], ascending=[True, False], na_position='last').reset_index(drop=True)
results_df

## 12) Best model per dataset

In [ ]:
ok_df = results_df[results_df['status'] == 'ok'].copy()
summary_df = ok_df.sort_values('test_acc', ascending=False).groupby('dataset_key', as_index=False).head(1)
summary_df[['dataset_label', 'task', 'model', 'test_acc', 'alpha', 'ERG_gap', 'generalization_gap']].sort_values('test_acc', ascending=False)

## 13) Plot set (3 plots per dataset)

In [ ]:
plot_dir = CHECKPOINT_DIR / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

ok_df = results_df[results_df['status'] == 'ok'].copy()
for c in ['alpha', 'ERG_gap', 'train_acc', 'test_acc', 'generalization_gap']:
    if c in ok_df.columns:
        ok_df[c] = pd.to_numeric(ok_df[c], errors='coerce')

for dataset_key, ddf in ok_df.groupby('dataset_key'):
    ddf = ddf.sort_values('test_acc', ascending=False)
    label = ddf['dataset_label'].iloc[0]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(ddf['alpha'], ddf['test_acc'], s=80)
    axes[0].set_xlabel('alpha')
    axes[0].set_ylabel('test accuracy')
    axes[0].set_title(f'{label}: test accuracy vs alpha')

    axes[1].scatter(ddf['alpha'], ddf['generalization_gap'], s=80)
    axes[1].set_xlabel('alpha')
    axes[1].set_ylabel('generalization gap')
    axes[1].set_title(f'{label}: generalization gap vs alpha')

    axes[2].scatter(ddf['ERG_gap'], ddf['test_acc'], s=80)
    axes[2].set_xlabel('ERG_gap')
    axes[2].set_ylabel('test accuracy')
    axes[2].set_title(f'{label}: test accuracy vs ERG_gap')

    plt.tight_layout()
    out = plot_dir / f'{dataset_key}_3plots.png'
    plt.savefig(out, dpi=140)
    plt.show()
    print('Saved:', out)

## 14) Optional: compact aggregate chart of best points

In [ ]:
if len(ok_df):
    best = ok_df.sort_values('test_acc', ascending=False).groupby('dataset_key', as_index=False).head(1)
    plt.figure(figsize=(10, 5))
    plt.barh(best['dataset_label'], best['test_acc'])
    plt.xlabel('Best test accuracy')
    plt.title('Best sweep accuracy per dataset')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()